In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer


# 1. Read the .csv file using Pandas. Take a look at the top few records.

In [2]:
df = pd.read_csv("k8_reviews_0.2.csv")
df.head()

,sentiment,review
0,1,Good but need updates and improvements
1,0,"Worst mobile i have bought ever, Battery is dr..."
2,1,when I will get my 10% cash back.... its alrea...
3,1,Good
4,0,The worst phone everThey have changed the last...


# 2. Normalize casings for the review text and extract the text into a list for easier manipulation.

In [3]:
df['review'] = df['review'].apply(lambda s: s.lower())
reviews = [str(s) for s in df['review']]
reviews

['good but need updates and improvements',
 "worst mobile i have bought ever, battery is draining like hell, backup is only 6 to 7 hours with internet uses, even if i put mobile idle its getting discharged.this is biggest lie from amazon & lenove which is not at all expected, they are making full by saying that battery is 4000mah & booster charger is fake, it takes at least 4 to 5 hours to be fully charged.don't know how lenovo will survive by making full of us.please don;t go for this else you will regret like me.",
 'when i will get my 10% cash back.... its already 15 january..',
 'good',
 'the worst phone everthey have changed the last phone but the problem is still same and the amazon is not returning the phone .highly disappointing of amazon',
 "only i'm telling don't buyi'm totally disappointedpoor batterypoor camerawaste of money",
 'phone is awesome. but while charging, it heats up allot..really a genuine reason to hate lenovo k8 note',
 'the battery level has worn down',
 "it'

# 3. Tokenize the reviews using NLTKs word_tokenize function.

In [4]:
from nltk.tokenize import word_tokenize
words = [word_tokenize(t) for t in reviews]
words

[['good', 'but', 'need', 'updates', 'and', 'improvements'],
 ['worst',
  'mobile',
  'i',
  'have',
  'bought',
  'ever',
  ',',
  'battery',
  'is',
  'draining',
  'like',
  'hell',
  ',',
  'backup',
  'is',
  'only',
  '6',
  'to',
  '7',
  'hours',
  'with',
  'internet',
  'uses',
  ',',
  'even',
  'if',
  'i',
  'put',
  'mobile',
  'idle',
  'its',
  'getting',
  'discharged.this',
  'is',
  'biggest',
  'lie',
  'from',
  'amazon',
  '&',
  'lenove',
  'which',
  'is',
  'not',
  'at',
  'all',
  'expected',
  ',',
  'they',
  'are',
  'making',
  'full',
  'by',
  'saying',
  'that',
  'battery',
  'is',
  '4000mah',
  '&',
  'booster',
  'charger',
  'is',
  'fake',
  ',',
  'it',
  'takes',
  'at',
  'least',
  '4',
  'to',
  '5',
  'hours',
  'to',
  'be',
  'fully',
  'charged.do',
  "n't",
  'know',
  'how',
  'lenovo',
  'will',
  'survive',
  'by',
  'making',
  'full',
  'of',
  'us.please',
  'don',
  ';',
  't',
  'go',
  'for',
  'this',
  'else',
  'you',
  'will

# 4. Perform parts-of-speech tagging on each sentence using the NLTK POS tagger.

In [8]:
from nltk.tag import pos_tag
import nltk
nltk.download('universal_tagset')
pos_tagged = [pos_tag(wordlist, tagset="universal", lang="eng") for wordlist in words]
pos_tagged

[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/millennium/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


[[('good', 'ADJ'),
  ('but', 'CONJ'),
  ('need', 'VERB'),
  ('updates', 'NOUN'),
  ('and', 'CONJ'),
  ('improvements', 'NOUN')],
 [('worst', 'ADJ'),
  ('mobile', 'NOUN'),
  ('i', 'NOUN'),
  ('have', 'VERB'),
  ('bought', 'VERB'),
  ('ever', 'ADV'),
  (',', '.'),
  ('battery', 'NOUN'),
  ('is', 'VERB'),
  ('draining', 'VERB'),
  ('like', 'ADP'),
  ('hell', 'NOUN'),
  (',', '.'),
  ('backup', 'NOUN'),
  ('is', 'VERB'),
  ('only', 'ADV'),
  ('6', 'NUM'),
  ('to', 'PRT'),
  ('7', 'NUM'),
  ('hours', 'NOUN'),
  ('with', 'ADP'),
  ('internet', 'ADJ'),
  ('uses', 'NOUN'),
  (',', '.'),
  ('even', 'ADV'),
  ('if', 'ADP'),
  ('i', 'ADJ'),
  ('put', 'VERB'),
  ('mobile', 'ADJ'),
  ('idle', 'NOUN'),
  ('its', 'PRON'),
  ('getting', 'VERB'),
  ('discharged.this', 'NOUN'),
  ('is', 'VERB'),
  ('biggest', 'ADJ'),
  ('lie', 'NOUN'),
  ('from', 'ADP'),
  ('amazon', 'NOUN'),
  ('&', 'CONJ'),
  ('lenove', 'NOUN'),
  ('which', 'DET'),
  ('is', 'VERB'),
  ('not', 'ADV'),
  ('at', 'ADP'),
  ('all', 'DET'),

# 5. For the topic model, we should want to include only nouns.

## 1. Find out all the POS tags that correspond to nouns.

In [35]:
postags = [[word[1] for word in wordlist] for wordlist in pos_tagged]
full_list = []
for wordlist in pos_tagged:
    full_list.extend([word[1] for word in wordlist])
full_list
tag_dict = dict.fromkeys(full_list)
tag_dict

{'ADJ': None,
 'CONJ': None,
 'VERB': None,
 'NOUN': None,
 'ADV': None,
 '.': None,
 'ADP': None,
 'NUM': None,
 'PRT': None,
 'PRON': None,
 'DET': None,
 'X': None}

### Observations
- Only NOUN and PRON apply here

## 2. Limit the data to only terms with these tags.

In [37]:
nouns = [[word[0] for word in wordlist if word[1] in ('NOUN', 'PRON')] for wordlist in pos_tagged]
nouns

[['updates', 'improvements'],
 ['mobile',
  'i',
  'battery',
  'hell',
  'backup',
  'hours',
  'uses',
  'idle',
  'its',
  'discharged.this',
  'lie',
  'amazon',
  'lenove',
  'they',
  'battery',
  'charger',
  'it',
  'hours',
  'don',
  'you',
  'me'],
 ['i', 'my', '%', 'cash', 'its', '..'],
 [],
 ['phone', 'everthey', 'phone', 'problem', 'amazon', 'phone', 'amazon'],
 ['camerawaste', 'money'],
 ['phone', 'it', 'allot', '..', 'reason', 'k8'],
 ['battery', 'level'],
 ['it',
  'problems',
  'phone',
  'hanging',
  'problems',
  'note',
  'station',
  'ahmedabad',
  'it',
  'years',
  'it',
  'phone',
  'lenovo'],
 ['lot', 'glitches', 'thing', 'options'],
 ['wrost'],
 ['phone', 'charger', 'damage', 'months'],
 ['item', 'it', 'battery', 'life'],
 ['i',
  'battery',
  'problem',
  'motherboard',
  'problem',
  'months',
  'mobile',
  'my',
  'life'],
 ['phone', 'slim', 'battry', 'backup', 'screen', 'it'],
 ['headset'],
 ['time', 'me', 'what', 'i'],
 ['product',
  'their',
  'prize',


# 6. Lemmatize.

In [42]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
lemmaWords = [[lemmatizer.lemmatize(word) for word in wordlist] for wordlist in nouns]
lemmaWords

[['update', 'improvement'],
 ['mobile',
  'i',
  'battery',
  'hell',
  'backup',
  'hour',
  'us',
  'idle',
  'it',
  'discharged.this',
  'lie',
  'amazon',
  'lenove',
  'they',
  'battery',
  'charger',
  'it',
  'hour',
  'don',
  'you',
  'me'],
 ['i', 'my', '%', 'cash', 'it', '..'],
 [],
 ['phone', 'everthey', 'phone', 'problem', 'amazon', 'phone', 'amazon'],
 ['camerawaste', 'money'],
 ['phone', 'it', 'allot', '..', 'reason', 'k8'],
 ['battery', 'level'],
 ['it',
  'problem',
  'phone',
  'hanging',
  'problem',
  'note',
  'station',
  'ahmedabad',
  'it',
  'year',
  'it',
  'phone',
  'lenovo'],
 ['lot', 'glitch', 'thing', 'option'],
 ['wrost'],
 ['phone', 'charger', 'damage', 'month'],
 ['item', 'it', 'battery', 'life'],
 ['i',
  'battery',
  'problem',
  'motherboard',
  'problem',
  'month',
  'mobile',
  'my',
  'life'],
 ['phone', 'slim', 'battry', 'backup', 'screen', 'it'],
 ['headset'],
 ['time', 'me', 'what', 'i'],
 ['product',
  'their',
  'prize',
  'range',
  'it

# 7. Remove stopwords and punctuation (if there are any).

In [ ]:
from nltk.corpus import stopwords
import string
STOPWORDS = set(stopwords.words('english'))
PUNCTUATION = set(string.punctuation)
cleaned = [[word for word in wordlist if word not in STOPWORDS and word not in PUNCTUATION] for wordlist  in lemmaWords]
cleaned = [wordlist for wordlist in cleaned if wordlist] # Remove reviews that have since become empty
cleaned

[['update', 'improvement'],
 ['mobile',
  'battery',
  'hell',
  'backup',
  'hour',
  'us',
  'idle',
  'discharged.this',
  'lie',
  'amazon',
  'lenove',
  'battery',
  'charger',
  'hour'],
 ['cash', '..'],
 ['phone', 'everthey', 'phone', 'problem', 'amazon', 'phone', 'amazon'],
 ['camerawaste', 'money'],
 ['phone', 'allot', '..', 'reason', 'k8'],
 ['battery', 'level'],
 ['problem',
  'phone',
  'hanging',
  'problem',
  'note',
  'station',
  'ahmedabad',
  'year',
  'phone',
  'lenovo'],
 ['lot', 'glitch', 'thing', 'option'],
 ['wrost'],
 ['phone', 'charger', 'damage', 'month'],
 ['item', 'battery', 'life'],
 ['battery', 'problem', 'motherboard', 'problem', 'month', 'mobile', 'life'],
 ['phone', 'slim', 'battry', 'backup', 'screen'],
 ['headset'],
 ['time'],
 ['product',
  'prize',
  'range',
  'specification',
  'comparison',
  'mobile',
  'range',
  'phone',
  'seal',
  'credit',
  'card',
  '..',
  '..',
  'deal',
  'amazon',
  '..'],
 ['battery', '..', 'solution', 'battery', 

# 8. Create a topic model using LDA on the cleaned-up data with 12 topics.

In [55]:
final_text = [' '.join(wordlist) for wordlist in cleaned]
final_text

['update improvement',
 'mobile battery hell backup hour us idle discharged.this lie amazon lenove battery charger hour',
 'cash ..',
 'phone everthey phone problem amazon phone amazon',
 'camerawaste money',
 'phone allot .. reason k8',
 'battery level',
 'problem phone hanging problem note station ahmedabad year phone lenovo',
 'lot glitch thing option',
 'wrost',
 'phone charger damage month',
 'item battery life',
 'battery problem motherboard problem month mobile life',
 'phone slim battry backup screen',
 'headset',
 'time',
 'product prize range specification comparison mobile range phone seal credit card .. .. deal amazon ..',
 'battery .. solution battery life',
 'smartphone',
 'galery problem speaker phone',
 'camera speed.excellent features.excelent battery',
 'product',
 'product camera battery phone product ..',
 'option cast screen wifi call option mobile hotspot',
 'phone usb cable',
 'phone price mobile lenovo display',
 'specification function phone any1 ..',
 'fon fon

In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidfVectorizer = TfidfVectorizer()
tfidfObject = tfidfVectorizer.fit_transform(cleaned)
tfIdfObject.shape

AttributeError: 'list' object has no attribute 'lower'

## 1. Print out the top terms for each topic.

## 2. What is the coherence of the model with the c_v metric?